In [1]:
import pandas as pd

In [3]:
df = pd.read_excel("C:/Users/hp/Downloads/a/05. Database RP May 2025 - AC REGISTER.xlsx", sheet_name="Raw", skiprows=1)

In [4]:
# Ganti 'df' dengan nama variabel dataframe kamu (misal df1 atau df)
print(df[['COCKPIT CREW TRAVEL', 'CABIN CREW TRAVEL']].head())

   COCKPIT CREW TRAVEL  CABIN CREW TRAVEL
0            77.115384          93.222903
1            92.312743          93.222903
2            73.858808          93.222903
3            82.543012          93.222903
4            76.029859          93.222903


In [ ]:
df = df.iloc[:, 1:]

In [ ]:
print([df])

In [4]:
df1 = df[
    (df['BLOCK HOURS'] > 0) & 
    (df['ASK (000)'] > 0) &
    (df['COCKPIT CREW TRAVEL'] >= 0) &
    (df['CABIN CREW TRAVEL'] >= 0)
].copy()

In [5]:
df_num = df.select_dtypes(include=["number"])

corr = df_num.corrwith(df_num['COCKPIT CREW TRAVEL']).sort_values(ascending=False).dropna()

print(corr.head(30))

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


COCKPIT CREW TRAVEL                  1.000000
TOTAL BO COSTS                       0.944138
CABIN CREW TRAVEL                    0.942900
TOTAL DIRECT,INDIRECT,FLEET COSTS    0.941632
BLOCK HOURS                          0.940685
FLIGHT HOURS                         0.940139
TOTAL COSTS                          0.939517
TOTAL DIRECT AND INDIRECT COSTS      0.939007
TOTAL DIRECT COSTS                   0.937859
TOTAL DIRECT FLIGHT COSTS            0.937174
FLIGHT KILOMETERS                    0.934466
ASK (000) C CLASS                    0.932360
FUEL BURN (IN LITER)                 0.929968
TOTAL FLEET COST                     0.928523
FUEL AIRCRAFT                        0.928174
LEASE AIRCRAFT                       0.919738
ATK (000)                            0.914568
TOTAL INDIRECT COSTS                 0.912438
COCKPIT CREW PERSON                  0.908387
ASK (000)                            0.901759
ATK PASSENGER (000)                  0.901577
MAINTENANCE RESERVE               

In [6]:
df_num = df.select_dtypes(include=["number"])

corr = df_num.corrwith(df_num['CABIN CREW TRAVEL']).sort_values(ascending=False).dropna()

print(corr.head(30))

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


CABIN CREW TRAVEL                    1.000000
ATK (000)                            0.975972
TOTAL DIRECT FLIGHT COSTS            0.974544
TOTAL DIRECT,INDIRECT,FLEET COSTS    0.974194
TOTAL DIRECT AND INDIRECT COSTS      0.974023
FUEL BURN (IN LITER)                 0.973903
TOTAL DIRECT COSTS                   0.973610
TOTAL BO COSTS                       0.973377
TOTAL COSTS                          0.972204
ATK PASSENGER (000)                  0.971816
ASK (000)                            0.971530
FUEL AIRCRAFT                        0.970106
ASK (000) Y CLASS                    0.966754
CABIN CREW PERSON                    0.953204
TOTAL FLEET COST                     0.951203
LEASE AIRCRAFT                       0.950823
COCKPIT CREW TRAVEL                  0.942900
ASK (000) C CLASS                    0.938959
TOTAL INDIRECT COSTS                 0.938527
FLIGHT KILOMETERS                    0.936724
MAINTENANCE RESERVE                  0.933494
FLIGHT HOURS                      

In [7]:
fitur = [
    'BLOCK HOURS',          
    'FLIGHT KILOMETERS',    
    'ASK (000)',            
    'NUMBER OF LANDING',    
    'AIRCRAFT TYPE',        
    'SERVICE TYPE',         
    'PERIODE',
    'ATK (000)', 
    'SEAT OFFERED',           
]

In [8]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor


In [9]:
def train_crew_model(target_name, df_data):
    print(f"--- TRAINING MODEL FOR: {target_name} ---")
    
    data = df_data[df_data[target_name] > 0].copy()
    
    X = data[fitur]
    y = data[target_name]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    categorical_cols = ['AIRCRAFT TYPE', 'SERVICE TYPE', 'PERIODE']
    numeric_cols = list(set(fitur) - set(categorical_cols))
    
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoder.fit(X_train[categorical_cols])
    
    X_train_enc = encoder.transform(X_train[categorical_cols])
    X_test_enc = encoder.transform(X_test[categorical_cols])
    enc_cols = encoder.get_feature_names_out(categorical_cols)
    
    X_train_final = pd.concat([
        X_train[numeric_cols].reset_index(drop=True),
        pd.DataFrame(X_train_enc, columns=enc_cols)
    ], axis=1)
    
    X_test_final = pd.concat([
        X_test[numeric_cols].reset_index(drop=True),
        pd.DataFrame(X_test_enc, columns=enc_cols)
    ], axis=1)
    
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        n_jobs=-1,
        objective="reg:absoluteerror" 
    )
    
    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)
    
    # Evaluasi
    mape = mean_absolute_percentage_error(y_test, y_pred)

    print(f"MAPE: {mape:.4f} ({mape*100:.2f}%)")
    
    return model, encoder, X_test_final, y_test, y_pred

In [10]:
bst_cockpit, enc_cockpit, X_test_cp, y_test_cp, y_pred_cp = train_crew_model('COCKPIT CREW TRAVEL', df1)

--- TRAINING MODEL FOR: COCKPIT CREW TRAVEL ---
MAPE: 0.0340 (3.40%)


In [11]:
bst_cabin, enc_cabin, X_test_cb, y_test_cb, y_pred_cb = train_crew_model('CABIN CREW TRAVEL', df1)


--- TRAINING MODEL FOR: CABIN CREW TRAVEL ---
MAPE: 0.0408 (4.08%)


In [12]:
print("\nCOCKPIT CREW TRAVEL")
results_cp = pd.DataFrame({'Actual': y_test_cp.values, 'Predicted': y_pred_cp})
results_cp['Diff'] = results_cp['Actual'] - results_cp['Predicted']
print(results_cp.head())
print("\nCOCKPIT CREW TRAVEL")
results_cb = pd.DataFrame({'Actual': y_test_cb.values, 'Predicted': y_pred_cb})
results_cb['Diff'] = results_cb['Actual'] - results_cb['Predicted']
print(results_cb.head())


COCKPIT CREW TRAVEL
       Actual   Predicted       Diff
0  182.585041  156.679657  25.905384
1  106.171559  119.615593 -13.444034
2  184.391236  184.417679  -0.026442
3  128.758906  129.485184  -0.726277
4  126.732800  123.699890   3.032910

COCKPIT CREW TRAVEL
       Actual   Predicted       Diff
0  395.460020  386.801361   8.658659
1  161.768471  173.225525 -11.457054
2  268.813794  277.358032  -8.544238
3  138.513795  134.575378   3.938417
4  157.169144  160.428909  -3.259765
